# Modelo 3 · Clustering: agrupar pingüinos sin etiquetas

**Objetivo:** encontrar **grupos naturales** en los datos **sin** una variable objetivo (aprendizaje **no supervisado**).

**Dataset:** *Palmer Penguins*, mediciones de pingüinos de la Antártida. Lo cargamos desde `seaborn`.

| Columna | Descripción |
|---|---|
| `bill_length_mm` | Largo del pico |
| `bill_depth_mm` | Profundidad del pico |
| `flipper_length_mm` | Largo de la aleta |
| `body_mass_g` | Masa corporal |
| `species` | Especie (**no** la usamos para entrenar, solo para verificar al final) |

**Diferencia clave:** en regresión y clasificación teníamos una "respuesta correcta" (`y`). Acá no: el algoritmo tiene que descubrir la estructura solo.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score

## 1. Cargar y limpiar los datos

In [ ]:
df = sns.load_dataset("penguins")
print(df.shape)
print(df.isna().sum())
df = df.dropna().reset_index(drop=True)
print(f"Después de eliminar faltantes: {df.shape}")
df.head()

## 2. Exploración (EDA)

Graficamos sin colorear por especie: ¿se ven grupos a simple vista?

In [ ]:
variables = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
sns.pairplot(df[variables], corner=True, plot_kws={"alpha": 0.6})
plt.show()

## 3. Escalar

K-Means usa **distancias**. Sin escalar, `body_mass_g` (miles) dominaría sobre `bill_depth_mm` (decenas).

In [ ]:
X = df[variables]
X_scaled = StandardScaler().fit_transform(X)
pd.DataFrame(X_scaled, columns=variables).describe().round(2)

## 4. ¿Cuántos grupos (k)?

Probamos varios valores de k con dos criterios:

- **Método del codo:** inercia (suma de distancias al centro del cluster). Buscamos donde la curva "se dobla".
- **Silhouette score:** qué tan separados están los clusters (de -1 a 1, más alto es mejor).

In [ ]:
ks = range(2, 9)
inercias, siluetas = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X_scaled)
    inercias.append(km.inertia_)
    siluetas.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(ks, inercias, "o-")
axes[0].set(title="Método del codo", xlabel="k", ylabel="Inercia")
axes[1].plot(ks, siluetas, "o-", color="tab:orange")
axes[1].set(title="Silhouette score", xlabel="k", ylabel="Silhouette")
plt.tight_layout()
plt.show()

> Los criterios no siempre coinciden: el silhouette favorece k=2 y el codo sugiere 2–3. Elegir k también depende del **conocimiento del dominio**. Usamos k=3.

## 5. Entrenar K-Means

In [ ]:
kmeans = KMeans(n_clusters=3, n_init=10, random_state=RANDOM_STATE)
df["cluster"] = kmeans.fit_predict(X_scaled)
df["cluster"].value_counts()

## 6. Visualizar

Con 4 variables no podemos graficar directamente. Usamos **PCA** para proyectar a 2 dimensiones.

In [ ]:
pca = PCA(n_components=2)
componentes = pca.fit_transform(X_scaled)
print(f"Varianza explicada por 2 componentes: {pca.explained_variance_ratio_.sum():.1%}")

plt.figure(figsize=(8, 6))
sns.scatterplot(x=componentes[:, 0], y=componentes[:, 1], hue=df["cluster"], palette="Set2", s=50)
plt.title("Clusters de K-Means (proyección PCA)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

## 7. Interpretar: ¿qué caracteriza a cada cluster?

In [ ]:
df.groupby("cluster")[variables].mean().round(1)

## 8. Verificar contra la especie real

En la vida real **no** tendríamos esta columna. Aquí la usamos para comprobar si los grupos que encontró el algoritmo tienen sentido.

- **Tabla cruzada:** cuántos pingüinos de cada especie cayeron en cada cluster.
- **Adjusted Rand Index (ARI):** 1 = coincidencia perfecta, 0 = azar.

In [ ]:
pd.crosstab(df["species"], df["cluster"])

In [ ]:
print(f"ARI: {adjusted_rand_score(df['species'], df['cluster']):.3f}")

## Conclusiones

- K-Means encontró grupos que coinciden en gran parte con las especies reales **sin haberlas visto nunca**.
- Escalar los datos es obligatorio en algoritmos basados en distancia.
- Elegir k no es automático: las métricas ayudan, pero hay que interpretar.

## Ejercicios

1. Repite el clustering **sin escalar** los datos. ¿Qué pasa con el ARI? ¿Por qué?
2. Prueba con k=2. ¿Qué especies quedan juntas? Mira el pairplot coloreado por `species` para entender por qué.
3. Agrega la variable `sex` (con one-hot encoding). ¿Los clusters ahora separan especies o sexos?
4. Prueba `AgglomerativeClustering` o `DBSCAN` de `sklearn.cluster` y compara el ARI con K-Means.